# FactorVAE: A Probabilistic Dynamic Factor Model Based on Variational Autoencoder for Predicting Cross-Sectional Stock Returns

**Authors:** Yitong Duan, Lei Wang, Qizhong Zhang, Jian Li

**Published:** 2022-06-28

[Read the paper](https://doi.org/10.1609/aaai.v36i4.20369)

**Abstract:**
As an asset pricing model in economics and finance, factor model has been widely used in quantitative investment. Towards building more effective factor models, recent years have witnessed the paradigm shift from linear models to more flexible nonlinear data-driven machine learning models. However, due to low signal-to-noise ratio of the financial data, it is quite challenging to learn effective factor models. In this paper, we propose a novel factor model, FactorVAE, as a probabilistic model with inherent randomness for noise modeling. Essentially, our model integrates the dynamic factor model (DFM) with the variational autoencoder (VAE) in machine learning, and we propose a prior-posterior learning method based on VAE, which can effectively guide the learning of model by approximating an optimal posterior factor model with future information. Particularly, considering that risk modeling is important for the noisy stock data, FactorVAE can estimate the variances from the distribution over the latent space of VAE, in addition to predicting returns. The experiments on the real stock market data demonstrate the effectiveness of FactorVAE, which outperforms various baseline methods.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration
TICKER = 'AAPL'
START_DATE = '2015-01-01'
END_DATE = '2023-01-01'
RISK_FREE_RATE = 0.01

## Phase 2: Data Download and Feature Engineering

In [ ]:
data = yf.download(TICKER, start=START_DATE, end=END_DATE)
data['Returns'] = data['Adj Close'].pct_change()
data.dropna(inplace=True)
data.head()

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Simple moving average crossover strategy
short_window = 40
long_window = 100

data['Short_MA'] = data['Adj Close'].rolling(window=short_window, min_periods=1).mean()
data['Long_MA'] = data['Adj Close'].rolling(window=long_window, min_periods=1).mean()
data['Signal'] = 0

data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1
data.loc[data['Short_MA'] <= data['Long_MA'], 'Signal'] = -1

## Phase 4: Vectorized Backtest

In [ ]:
# Shift signals to avoid look-ahead bias
data['Strategy_Returns'] = data['Signal'].shift(1) * data['Returns']
data.dropna(inplace=True)
data['Cumulative_Strategy_Returns'] = (1 + data['Strategy_Returns']).cumprod()
data['Cumulative_Market_Returns'] = (1 + data['Returns']).cumprod()

## Phase 5: Performance Metrics and Visualization

In [ ]:
def calculate_performance_metrics(returns, risk_free_rate):
    excess_returns = returns - risk_free_rate / 252
    sharpe_ratio = np.mean(excess_returns) / np.std(excess_returns) * np.sqrt(252)
    downside_returns = excess_returns[excess_returns < 0]
    sortino_ratio = np.mean(excess_returns) / np.std(downside_returns) * np.sqrt(252)
    drawdown = data['Cumulative_Strategy_Returns'].div(data['Cumulative_Strategy_Returns'].cummax()).sub(1)
    max_drawdown = drawdown.min()
    calmar_ratio = np.mean(excess_returns) * 252 / abs(max_drawdown)
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

sharpe, sortino, calmar, max_dd = calculate_performance_metrics(data['Strategy_Returns'], RISK_FREE_RATE)
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"Max Drawdown: {max_dd:.2%}")

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative_Strategy_Returns'], label='Strategy')
plt.plot(data['Cumulative_Market_Returns'], label='Market')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

## Phase 6: Monitoring Stub

In [ ]:
# Placeholder for monitoring logic
# This could include real-time data fetching, alerting, etc.
print('Monitoring phase is not implemented yet.')